In [ ]:
import json

import pandas as pd
from IPython import get_ipython

# --- Agent loop: accumulate per-cell previews; final cell calls dbutils.notebook.exit ---
_AGENT_REPORT = {
    "ok": True,
    "message": "",
    "table": None,
    "preview": None,
    "previews": {},
}
_PREVIEW_MAX_ROWS = 5
_PREVIEW_MAX_COLS = 30
_PREVIEW_MAX_STR = 800


def _preview_dataframe_like(obj):
    """Return {columns, rows} for pandas or Spark DataFrames, else None."""
    try:
        if hasattr(obj, "limit") and hasattr(obj, "toPandas"):  # Spark DataFrame
            pdf = obj.limit(_PREVIEW_MAX_ROWS).toPandas()
        elif isinstance(obj, pd.DataFrame):
            pdf = obj.head(_PREVIEW_MAX_ROWS).copy()
        else:
            return None
        cols = list(pdf.columns)[:_PREVIEW_MAX_COLS]
        pdf = pdf[cols]
        rows = json.loads(pdf.astype(str).to_json(orient="records"))
        truncated = False
        if isinstance(obj, pd.DataFrame):
            truncated = len(obj) > _PREVIEW_MAX_ROWS or len(obj.columns) > len(cols)
        elif hasattr(obj, "count"):
            try:
                truncated = obj.count() > _PREVIEW_MAX_ROWS
            except Exception:
                truncated = True
        return {"columns": cols, "rows": rows, "truncated": truncated}
    except Exception as exc:  # noqa: BLE001
        return {"error": str(exc)[:_PREVIEW_MAX_STR]}


def _post_run_cell(result):
    global _AGENT_REPORT
    try:
        ip = get_ipython()
        if ip is None:
            return
        label = f"cell_{ip.execution_count}"
        if result is None:
            return
        err = getattr(result, "error_in_exec", None)
        if err:
            _AGENT_REPORT["ok"] = False
            _AGENT_REPORT["previews"][label] = {
                "type": "error",
                "message": str(err)[:_PREVIEW_MAX_STR],
            }
            return
        value = getattr(result, "result", None)
        if value is None:
            return
        prev = _preview_dataframe_like(value)
        if prev is not None and "error" not in prev:
            _AGENT_REPORT["previews"][label] = {"type": "dataframe", **prev}
        else:
            text = repr(value)
            if len(text) > _PREVIEW_MAX_STR:
                text = text[:_PREVIEW_MAX_STR] + "…"
            _AGENT_REPORT["previews"][label] = {"type": "repr", "value": text}
    except Exception as exc:  # noqa: BLE001
        _AGENT_REPORT["previews"]["hook_error"] = {"type": "error", "message": str(exc)[:_PREVIEW_MAX_STR]}


_ip = get_ipython()
if _ip is not None and not getattr(_ip, "_agent_report_hook_registered", False):
    _ip.events.register("post_run_cell", _post_run_cell)
    _ip._agent_report_hook_registered = True


In [ ]:
# Read from Unity Catalog table and convert to pandas
patient_data_df = spark.table('workspace.default.diabetic_data').toPandas()

In [ ]:
patient_data_df.head()

In [ ]:
from pyspark.sql import functions as F

# Bronze source
bronze_df = spark.table("workspace.default.diabetic_data")

silver_df = (
    bronze_df
    .select(
        F.col("patient_nbr").cast("string").alias("patient_id"),
        F.col("encounter_id").cast("string").alias("encounter_id"),
        F.col("age").cast("string").alias("age"),
        F.col("time_in_hospital").alias("time_in_hospital"),
        F.col("readmitted").cast("string").alias("readmitted"),
        F.col("diag_1").cast("string").alias("diag_1"),
        F.col("admission_type_id").alias("admission_type_id"),
        F.col("discharge_disposition_id").alias("discharge_disposition_id"),
    )
    .withColumn("patient_id", F.trim("patient_id"))
    .withColumn("encounter_id", F.trim("encounter_id"))
    .withColumn("age", F.trim("age"))
    .withColumn("readmitted", F.upper(F.trim("readmitted")))
    .withColumn("diag_1", F.trim("diag_1"))
    .withColumn("patient_id", F.when((F.col("patient_id") == "") | (F.col("patient_id") == "?"), None).otherwise(F.col("patient_id")))
    .withColumn("encounter_id", F.when((F.col("encounter_id") == "") | (F.col("encounter_id") == "?"), None).otherwise(F.col("encounter_id")))
    .withColumn("age", F.when((F.col("age") == "") | (F.col("age") == "?"), None).otherwise(F.col("age")))
    .withColumn("diag_1", F.when((F.col("diag_1") == "") | (F.col("diag_1") == "?"), None).otherwise(F.col("diag_1")))
    .withColumn(
        "age",
        F.when(F.col("age").isNull(), None)
         .otherwise(F.regexp_replace(F.regexp_replace(F.col("age"), r"^\[", ""), r"\)$", ""))
    )
    .withColumn(
        "readmitted",
        F.when(F.col("readmitted").isin("NO", "<30", ">30"), F.col("readmitted")).otherwise(None)
    )
    .withColumn("time_in_hospital", F.col("time_in_hospital").cast("int"))
    .withColumn("admission_type_id", F.col("admission_type_id").cast("int"))
    .withColumn("discharge_disposition_id", F.col("discharge_disposition_id").cast("int"))
    .filter(F.col("patient_id").isNotNull())
    .filter(F.col("encounter_id").isNotNull())
    .filter(F.col("time_in_hospital").isNotNull() & (F.col("time_in_hospital") >= 0))
    .filter(F.col("admission_type_id").isNotNull() & (F.col("admission_type_id") > 0))
    .filter(F.col("discharge_disposition_id").isNotNull() & (F.col("discharge_disposition_id") > 0))
    .dropDuplicates(["encounter_id"])
)

In [ ]:
silver_row_count = silver_df.count()
print(f"Silver dataset row count: {silver_row_count}")
print(f"Silver dataset column count: {len(silver_df.columns)}")

In [ ]:
silver_df.head(10)

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.patient_encounters")
)

spark.sql("SELECT COUNT(*) AS row_count FROM workspace.silver.patient_encounters").show()
spark.sql("SELECT DISTINCT readmitted FROM workspace.silver.patient_encounters ORDER BY readmitted").show()
spark.sql("SELECT DISTINCT age FROM workspace.silver.patient_encounters ORDER BY age LIMIT 20").show(truncate=False)
spark.sql("DESCRIBE TABLE workspace.silver.patient_encounters").show(truncate=False)

In [ ]:
ids_mapping_df = spark.table('workspace.default.ids_mapping').toPandas()

In [ ]:
ids_mapping_df

In [ ]:
for row in ids_mapping_df.iterrows():
    print(row)

In [ ]:
import pandas as pd
from pyspark.sql import functions as F

# Cleanup raw ids_mapping into 3 clean mapping datasets
ids_mapping_pdf = spark.table("workspace.default.ids_mapping").toPandas()
ids_mapping_pdf = ids_mapping_pdf.rename(columns={"admission_type_id": "raw_id", "description": "description"})
ids_mapping_pdf["raw_id"] = ids_mapping_pdf["raw_id"].astype("string").str.strip()
ids_mapping_pdf["description"] = ids_mapping_pdf["description"].astype("string").str.strip()

section_switch = {
    "discharge_disposition_id": "discharge_disposition",
    "admission_source_id": "admission_source",
}

current_section = "admission_type"
admission_type_rows = []
discharge_disposition_rows = []
admission_source_rows = []

for _, row in ids_mapping_pdf.iterrows():
    raw_id = None if pd.isna(row["raw_id"]) else str(row["raw_id"]).strip()
    desc = None if pd.isna(row["description"]) else str(row["description"]).strip()

    # Skip empty separators and stringified null rows
    if raw_id in (None, "", "None", "<NA>") and desc in (None, "", "None", "<NA>"):
        continue

    # Section header rows embedded in data
    if raw_id in section_switch and desc == "description":
        current_section = section_switch[raw_id]
        continue

    # Skip malformed/header-like rows
    if raw_id in ("admission_type_id", "description", "discharge_disposition_id", "admission_source_id"):
        continue
    if not str(raw_id).isdigit():
        continue

    rec = {"id": int(raw_id), "description": desc if desc not in ("<NA>", "None") else None}

    if current_section == "admission_type":
        admission_type_rows.append(rec)
    elif current_section == "discharge_disposition":
        discharge_disposition_rows.append(rec)
    elif current_section == "admission_source":
        admission_source_rows.append(rec)

admission_type_pdf = pd.DataFrame(admission_type_rows).drop_duplicates(subset=["id"]).sort_values("id")
discharge_disposition_pdf = pd.DataFrame(discharge_disposition_rows).drop_duplicates(subset=["id"]).sort_values("id")
admission_source_pdf = pd.DataFrame(admission_source_rows).drop_duplicates(subset=["id"]).sort_values("id")

admission_type_df = (
    spark.createDataFrame(admission_type_pdf)
    .withColumnRenamed("id", "admission_type_id")
    .withColumn("admission_type_id", F.col("admission_type_id").cast("int"))
    .withColumn("description", F.col("description").cast("string"))
)

discharge_disposition_df = (
    spark.createDataFrame(discharge_disposition_pdf)
    .withColumnRenamed("id", "discharge_disposition_id")
    .withColumn("discharge_disposition_id", F.col("discharge_disposition_id").cast("int"))
    .withColumn("description", F.col("description").cast("string"))
)

admission_source_df = (
    spark.createDataFrame(admission_source_pdf)
    .withColumnRenamed("id", "admission_source_id")
    .withColumn("admission_source_id", F.col("admission_source_id").cast("int"))
    .withColumn("description", F.col("description").cast("string"))
)

In [ ]:
# Verify cleaned mapping outputs before writing
print("admission_type rows:", admission_type_df.count())
print("discharge_disposition rows:", discharge_disposition_df.count())
print("admission_source rows:", admission_source_df.count())

print("\nDistinct ID counts (should match row counts):")
print("admission_type distinct IDs:", admission_type_df.select("admission_type_id").distinct().count())
print("discharge_disposition distinct IDs:", discharge_disposition_df.select("discharge_disposition_id").distinct().count())
print("admission_source distinct IDs:", admission_source_df.select("admission_source_id").distinct().count())

print("\nSample rows:")
admission_type_df.orderBy("admission_type_id").show(10, truncate=False)
discharge_disposition_df.orderBy("discharge_disposition_id").show(10, truncate=False)
admission_source_df.orderBy("admission_source_id").show(10, truncate=False)

In [ ]:
# Write cleaned mapping tables to silver and read back to verify
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

admission_type_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.admission_type_mapping")
discharge_disposition_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.discharge_disposition_mapping")
admission_source_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.admission_source_mapping")

print("Written tables:")
print("- workspace.silver.admission_type_mapping")
print("- workspace.silver.discharge_disposition_mapping")
print("- workspace.silver.admission_source_mapping")

print("\nRead-back verification:")
spark.sql("SELECT COUNT(*) AS row_count FROM workspace.silver.admission_type_mapping").show()
spark.sql("SELECT COUNT(*) AS row_count FROM workspace.silver.discharge_disposition_mapping").show()
spark.sql("SELECT COUNT(*) AS row_count FROM workspace.silver.admission_source_mapping").show()

spark.sql("SELECT * FROM workspace.silver.admission_type_mapping ORDER BY admission_type_id LIMIT 10").show(truncate=False)
spark.sql("SELECT * FROM workspace.silver.discharge_disposition_mapping ORDER BY discharge_disposition_id LIMIT 10").show(truncate=False)
spark.sql("SELECT * FROM workspace.silver.admission_source_mapping ORDER BY admission_source_id LIMIT 10").show(truncate=False)

In [ ]:
from pyspark.sql import functions as F

# Build diagnoses-focused silver table for readmission analysis
bronze_diag_df = spark.table("workspace.default.diabetic_data")

diagnoses_count_df = (
    bronze_diag_df
    .select(
        F.col("patient_nbr").cast("string").alias("patient_nbr"),
        F.col("diag_1").cast("string").alias("diag_1"),
        F.col("diag_2").cast("string").alias("diag_2"),
        F.col("diag_3").cast("string").alias("diag_3"),
        F.col("readmitted").cast("string").alias("readmission"),
    )
    .withColumn("patient_nbr", F.trim("patient_nbr"))
    .withColumn("diag_1", F.trim("diag_1"))
    .withColumn("diag_2", F.trim("diag_2"))
    .withColumn("diag_3", F.trim("diag_3"))
    .withColumn("readmission", F.upper(F.trim("readmission")))
    .withColumn("patient_nbr", F.when((F.col("patient_nbr") == "") | (F.col("patient_nbr") == "?"), None).otherwise(F.col("patient_nbr")))
    .withColumn("diag_1", F.when((F.col("diag_1") == "") | (F.col("diag_1") == "?"), None).otherwise(F.col("diag_1")))
    .withColumn("diag_2", F.when((F.col("diag_2") == "") | (F.col("diag_2") == "?"), None).otherwise(F.col("diag_2")))
    .withColumn("diag_3", F.when((F.col("diag_3") == "") | (F.col("diag_3") == "?"), None).otherwise(F.col("diag_3")))
    .withColumn(
        "readmission",
        F.when(F.col("readmission").isin("NO", "<30", ">30"), F.col("readmission")).otherwise(None),
    )
    .withColumn(
        "num_diag",
        F.col("diag_1").isNotNull().cast("int")
        + F.col("diag_2").isNotNull().cast("int")
        + F.col("diag_3").isNotNull().cast("int"),
    )
    .filter(F.col("patient_nbr").isNotNull())
)

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
(
    diagnoses_count_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.diagnoses_count")
)

spark.sql("SELECT COUNT(*) AS row_count FROM workspace.silver.diagnoses_count").show()
spark.sql("SELECT num_diag, COUNT(*) AS rows FROM workspace.silver.diagnoses_count GROUP BY num_diag ORDER BY num_diag").show()

# Readmission likelihood by number of available diagnosis fields
spark.sql("""
SELECT
  num_diag,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN readmission IN ('<30', '>30') THEN 1 ELSE 0 END) AS readmitted_rows,
  ROUND(100.0 * SUM(CASE WHEN readmission IN ('<30', '>30') THEN 1 ELSE 0 END) / COUNT(*), 2) AS readmission_rate_pct
FROM workspace.silver.diagnoses_count
GROUP BY num_diag
ORDER BY num_diag
""").show()

spark.sql("SELECT * FROM workspace.silver.diagnoses_count LIMIT 10").show(truncate=False)

In [ ]:
# Duplicate diagnoses_count build removed.
# Keep a single canonical build/write cell above to avoid redundant table overwrites.

In [ ]:
# Removed temporary validation cell.

In [ ]:
# Quantify readmission likelihood by diagnosis count
spark.sql("""
SELECT
  num_diag,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN readmission IN ('<30', '>30') THEN 1 ELSE 0 END) AS readmitted_rows,
  ROUND(100.0 * SUM(CASE WHEN readmission IN ('<30', '>30') THEN 1 ELSE 0 END) / COUNT(*), 2) AS readmission_rate_pct
FROM workspace.silver.diagnoses_count
GROUP BY num_diag
ORDER BY num_diag
""").show()

In [ ]:
# --- Agent report: single JSON payload for `databricks jobs get-run-output` ---
import json

_AGENT_REPORT["message"] = (
    "Notebook finished. See `previews` for per-cell execution results (dataframe heads / reprs). "
    "Diagnoses readmission analysis table created: workspace.silver.diagnoses_count (with num_diag derived from diag_1/diag_2/diag_3); "
    "other silver outputs include patient_encounters and the three mapping tables."
)
_AGENT_REPORT["table"] = "workspace.silver.diagnoses_count"

# Optional top-level `preview`: first dataframe preview we captured (handy for demos)
_AGENT_REPORT["preview"] = None
for _k in sorted(_AGENT_REPORT.get("previews", {})):
    _v = _AGENT_REPORT["previews"][_k]
    if isinstance(_v, dict) and _v.get("type") == "dataframe":
        _AGENT_REPORT["preview"] = {"source_cell": _k, "columns": _v.get("columns"), "rows": _v.get("rows")}
        break

_payload = json.dumps(_AGENT_REPORT)
if len(_payload) > 4_500_000:
    _keys = list(_AGENT_REPORT["previews"])
    _AGENT_REPORT["previews"] = {k: _AGENT_REPORT["previews"][k] for k in _keys[-15:]}
    _AGENT_REPORT["message"] += " (previews truncated to last 15 cells)"
    _payload = json.dumps(_AGENT_REPORT)

dbutils.notebook.exit(_payload)
